In [1]:
import numpy as np
import tensorflow as tf
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import pickle
import os

import pandas as pd

2025-05-20 08:11:36.121918: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747728696.313393      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747728696.379423      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
df = pd.read_csv("/kaggle/input/news-1/news.csv")
df = df[['Title', 'Sentiment']].dropna()
sentiment_map = {'отрицательный': 0, 'нейтральный': 1, 'положительный': 2}
df['label'] = df['Sentiment'].map(sentiment_map)

In [3]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['Title'].tolist(), df['label'].tolist(), test_size=0.3, random_state=42)

tokenizer = Tokenizer(num_words=20000, lower=True)
tokenizer.fit_on_texts(train_texts)
vocab_size = len(tokenizer.word_index) + 1
max_len = 45

def preprocess(texts, labels):
    seqs = tokenizer.texts_to_sequences(texts)
    padded = pad_sequences(seqs, maxlen=max_len)
    docs = np.expand_dims(padded, axis=1)  # 1 sentence per document
    labels = tf.keras.utils.to_categorical(labels, num_classes=3)
    return docs, labels

x_train, y_train = preprocess(train_texts, train_labels)
x_test, y_test = preprocess(test_texts, test_labels)

Model

In [9]:
import numpy as np

def length(sequences):
    used = tf.cast(tf.math.not_equal(sequences, 0), tf.float32)
    seq_len = tf.reduce_sum(used, axis=1)
    return tf.cast(seq_len, tf.int32)

def read_question():
    with open('q1_data', 'rb') as f:
        q1 = pickle.load(f)
    with open('q2_data', 'rb') as g:
        q2 = pickle.load(g)
    with open('q3_data', 'rb') as b:
        q3 = pickle.load(b)
    return [q1, q2, q3]

def shuffle_data(x, y):
    indices = np.random.permutation(len(x))
    return [x[i] for i in indices], [y[i] for i in indices]

class FISHQA(tf.keras.Model):
    def __init__(self, vocab_size, num_classes, embedding_size=200, hidden_size=64, dropout_keep_proba=0.5, query=[]):
        super(FISHQA, self).__init__()
        self.vocab_size = vocab_size
        self.num_classes = num_classes
        self.embedding_size = embedding_size
        self.hidden_size = hidden_size
        self.dropout_keep_proba = dropout_keep_proba
        self.query = query

        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_size)
        self.query_projector = tf.keras.layers.Dense(hidden_size * 2, activation='tanh')

        self.gru_fw = tf.keras.layers.GRU(hidden_size, return_sequences=True)
        self.gru_bw = tf.keras.layers.GRU(hidden_size, return_sequences=True, go_backwards=True)
        self.dropout = tf.keras.layers.Dropout(rate=1 - dropout_keep_proba)
        self.classifier = tf.keras.layers.Dense(num_classes)
        self.context_vector = self.add_weight(
            name="context_vector",
            shape=(hidden_size * 2,),
            initializer=tf.keras.initializers.TruncatedNormal(stddev=0.1),
            trainable=True
        )

        self.att_dense_1 = tf.keras.layers.Dense(hidden_size * 2, activation='tanh')
        self.att_dense_2 = tf.keras.layers.Dense(hidden_size * 2, activation='tanh')
        self.att_dense_3 = tf.keras.layers.Dense(hidden_size * 2, activation='tanh')
        self.att_dense_4 = tf.keras.layers.Dense(hidden_size * 2, activation='tanh')

    def build(self, input_shape):
        B, S, F = input_shape
        dummy_input = tf.zeros((B, S, F))
        self.call(dummy_input, training=False)
        super().build(input_shape)

    def call(self, input_x, training=False):
        B = tf.shape(input_x)[0]
        S = tf.shape(input_x)[1]
        W = tf.shape(input_x)[2]

        x = tf.reshape(input_x, [-1, W])
        embedded = self.embedding(x)

        q1_emb = self._query_embedding(self.query[0])
        q2_emb = self._query_embedding(self.query[1])
        q3_emb = self._query_embedding(self.query[2])

        word_encoded = self._bi_gru(embedded)
        sent_vec, _ = self._attention(word_encoded, q1_emb, q2_emb, q3_emb)

        sent_vec = self.dropout(sent_vec, training=training)
        sent_vec = tf.reshape(sent_vec, [B, S, self.hidden_size * 2])

        doc_encoded = self._bi_gru(sent_vec)
        doc_vec, _ = self._attention(doc_encoded, q1_emb, q2_emb, q3_emb)

        doc_vec = self.dropout(doc_vec, training=training)
        logits = self.classifier(doc_vec)
        return logits

    def _query_embedding(self, query):
        emb = self.embedding(tf.constant(query, dtype=tf.int32))  # [seq_len, emb_dim]
        avg = tf.reduce_mean(emb, axis=0)  # [emb_dim]
        avg = tf.expand_dims(avg, axis=0)  # [1, emb_dim] to make it 2D for Dense
        projected = self.query_projector(avg)  # [1, hidden_size * 2]
        return tf.squeeze(projected, axis=0)   # back to [hidden_size * 2]

    
    def _bi_gru(self, inputs):
        fw = self.gru_fw(inputs)
        bw = self.gru_bw(inputs)
        return tf.concat([fw, bw], axis=-1)

    def _attention(self, inputs, q1, q2, q3):
        context = self.context_vector

        h1 = self.att_dense_1(inputs)
        h2 = self.att_dense_2(inputs)
        h3 = self.att_dense_3(inputs)
        h4 = self.att_dense_4(inputs)

        def compute_alpha(h, query):
            query = tf.reshape(query, [1, 1, -1])
            query = tf.broadcast_to(query, tf.shape(h))
            score = tf.reduce_sum(h * query, axis=-1, keepdims=True)
            return tf.nn.softmax(score, axis=1)

        t_alpha = compute_alpha(h1, context)
        q_alpha1 = compute_alpha(h2, q1)
        q_alpha2 = compute_alpha(h3, q2)
        q_alpha3 = compute_alpha(h4, q3)

        alpha = (t_alpha + q_alpha1 + q_alpha2 + q_alpha3) / 4
        output = tf.reduce_sum(inputs * alpha, axis=1)
        return output, alpha

In [5]:
q1 = [
    'долг', 'обязательства', 'кредит', 'дефолт', 'невыплата',
    'неплатеж', 'ипотека', 'рефинансирование',
    'просрочка', 'финансовая нестабильность', 'платежный кризис',
    'дефицит', 'снижение ликвидности', 'отказ от выплаты'
]

q2 = [
    'увольнение', 'смена руководства', 'отставка', 'кадровые перестановки',
    'сокращение персонала', 'реорганизация',
    'назначение нового директора', 'расформирование отдела', 'ротация сотрудников', 'массовые увольнения'
]

q3 = [
    'суд', 'расследование', 'штраф', 'иск', 'обвинение',
    'нарушение закона', 'уголовное дело', 'следствие', 'взыскание',
    'регуляторное давление', 'финансовая проверка', 'контроль ФНС',
    'антикоррупционное дело', 'арест', 'рассмотрение дела'
]

# Tokenize and pad queries (each must be 45 long for averaging)
def encode_query(q):
    tokens = tokenizer.texts_to_sequences(q)
    flat = [tok[0] if tok else 0 for tok in tokens]
    return flat + [0] * (45 - len(flat))

q1_data = encode_query(q1)
q2_data = encode_query(q2)
q3_data = encode_query(q3)

# Save them for model to load
with open("q1_data", "wb") as f: pickle.dump(q1_data, f)
with open("q2_data", "wb") as f: pickle.dump(q2_data, f)
with open("q3_data", "wb") as f: pickle.dump(q3_data, f)

query_set = read_question()

In [10]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.set_visible_devices(gpus[0], 'GPU')
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPUs Available: {[gpu.name for gpu in gpus]}")
    except RuntimeError as e:
        print(e)

# Model instantiation
model = FISHQA(
    vocab_size=vocab_size,
    num_classes=3,
    embedding_size=200,
    hidden_size=64,
    dropout_keep_proba=0.5,
    query=query_set
)

optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
loss_fn = tf.keras.losses.CategoricalCrossentropy(from_logits=True)

model.compile(optimizer=optimizer, loss=loss_fn)
model.build(input_shape=(64, 1, 45))
model.load_weights("/kaggle/input/finalfinal/fishqa_epoch10.weights.h5")

GPUs Available: ['/physical_device:GPU:0', '/physical_device:GPU:1']


In [11]:
epochs = 5
batch_size = 64

train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).shuffle(buffer_size=1024).batch(batch_size)
test_dataset = tf.data.Dataset.from_tensor_slices((x_test, y_test)).batch(batch_size)

y_true = []
y_pred = []
for x_batch, y_batch in test_dataset:
    logits = model(x_batch, training=False)
    predictions = tf.argmax(logits, axis=1)
    labels = tf.argmax(y_batch, axis=1)
    y_true.extend(labels.numpy())
    y_pred.extend(predictions.numpy())

acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average='macro')
print(f"Accuracy: {acc:.4f} - F1 Score: {f1:.4f}")

Accuracy: 0.8190 - F1 Score: 0.8097


In [ ]:
epochs = 5
batch_size = 64

train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).shuffle(buffer_size=1024).batch(batch_size)
test_dataset = tf.data.Dataset.from_tensor_slices((x_test, y_test)).batch(batch_size)

for epoch in range(1, epochs + 1):
    print(f"\nEpoch {epoch}/{epochs}")
    train_loss = tf.keras.metrics.Mean()
    
    # Training
    for x_batch, y_batch in tqdm(train_dataset, desc="Training"):
        with tf.GradientTape() as tape:
            logits = model(x_batch, training=True)
            loss = loss_fn(y_batch, logits)
        gradients = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(gradients, model.trainable_variables))
        train_loss.update_state(loss)

    # Evaluation every 5 epochs
    if epoch % 5 == 0 or epoch == epochs:
        y_true = []
        y_pred = []
        for x_batch, y_batch in test_dataset:
            logits = model(x_batch, training=False)
            predictions = tf.argmax(logits, axis=1)
            labels = tf.argmax(y_batch, axis=1)
            y_true.extend(labels.numpy())
            y_pred.extend(predictions.numpy())
        
        acc = accuracy_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred, average='macro')
        print(f"[Epoch {epoch}] Loss: {train_loss.result().numpy():.4f} - Accuracy: {acc:.4f} - F1 Score: {f1:.4f}")

In [ ]:
# saving the model
model_save_path = f"./fishqa_model_5"
model.save_weights(f"/kaggle/working/fishqa_epoch5.weights.h5")
print(f"\nModel saved to: {model_save_path}")

In [ ]:
epochs = 5
batch_size = 64

train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).shuffle(buffer_size=1024).batch(batch_size)
test_dataset = tf.data.Dataset.from_tensor_slices((x_test, y_test)).batch(batch_size)

for epoch in range(1, epochs + 1):
    print(f"\nEpoch {epoch}/{epochs}")
    train_loss = tf.keras.metrics.Mean()
    
    # Training
    for x_batch, y_batch in tqdm(train_dataset, desc="Training"):
        with tf.GradientTape() as tape:
            logits = model(x_batch, training=True)
            loss = loss_fn(y_batch, logits)
        gradients = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(gradients, model.trainable_variables))
        train_loss.update_state(loss)

    # Evaluation every 5 epochs
    if epoch % 5 == 0 or epoch == epochs:
        y_true = []
        y_pred = []
        for x_batch, y_batch in test_dataset:
            logits = model(x_batch, training=False)
            predictions = tf.argmax(logits, axis=1)
            labels = tf.argmax(y_batch, axis=1)
            y_true.extend(labels.numpy())
            y_pred.extend(predictions.numpy())
        
        acc = accuracy_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred, average='macro')
        print(f"[Epoch {epoch}] Loss: {train_loss.result().numpy():.4f} - Accuracy: {acc:.4f} - F1 Score: {f1:.4f}")

# saving the model
model_save_path = f"./fishqa_model_5"
model.save_weights(f"/kaggle/working/fishqa_epoch5.weights.h5")
print(f"\nModel saved to: {model_save_path}")

In [ ]:
epochs = 5
batch_size = 64

train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).shuffle(buffer_size=1024).batch(batch_size)
test_dataset = tf.data.Dataset.from_tensor_slices((x_test, y_test)).batch(batch_size)

for epoch in range(1, epochs + 1):
    print(f"\nEpoch {epoch}/{epochs}")
    train_loss = tf.keras.metrics.Mean()
    
    # Training
    for x_batch, y_batch in tqdm(train_dataset, desc="Training"):
        with tf.GradientTape() as tape:
            logits = model(x_batch, training=True)
            loss = loss_fn(y_batch, logits)
        gradients = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(gradients, model.trainable_variables))
        train_loss.update_state(loss)

    # Evaluation every 5 epochs
    if epoch % 5 == 0 or epoch == epochs:
        y_true = []
        y_pred = []
        for x_batch, y_batch in test_dataset:
            logits = model(x_batch, training=False)
            predictions = tf.argmax(logits, axis=1)
            labels = tf.argmax(y_batch, axis=1)
            y_true.extend(labels.numpy())
            y_pred.extend(predictions.numpy())
        
        acc = accuracy_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred, average='macro')
        print(f"[Epoch {epoch}] Loss: {train_loss.result().numpy():.4f} - Accuracy: {acc:.4f} - F1 Score: {f1:.4f}")

# saving the model
model_save_path = f"./fishqa_model_5"
model.save_weights(f"/kaggle/working/fishqa_epoch5.weights.h5")
print(f"\nModel saved to: {model_save_path}")